<a href="https://colab.research.google.com/github/CMILINAZZO/medical-llm-self-bias-audit/blob/main/notebooks/01_data_slicing_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: Data Slicing and Verification

In [ ]:
# Install Hugging Face datasets library
#!pip install datasets pandas

In [3]:
from datasets import load_dataset
import pandas as pd

# 1. Load the expert-annotated subset of PubMedQA
raw_dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

# 2. Convert to a standard Pandas DataFrame
df_raw = pd.DataFrame(raw_dataset)

# 3. Quick structural check
print(f"Successfully loaded {len(df_raw)} raw rows.")
print("Available columns:", df_raw.columns.tolist())
df_raw.head(2)

Successfully loaded 1000 raw rows.
Available columns: ['pubid', 'question', 'context', 'long_answer', 'final_decision']


,pubid,question,context,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lac...,{'contexts': ['Programmed cell death (PCD) is ...,Results depicted mitochondrial dynamics in viv...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,{'contexts': ['Assessment of visual acuity dep...,"Using the charts described, there was only a s...",no


In [4]:
# Function to stitch the list of context paragraphs into one clean string
def process_medical_context(context_obj):
    if isinstance(context_obj, dict) and 'contexts' in context_obj:
        return " ".join(context_obj['contexts'])
    return str(context_obj)

# Apply context cleaning
df_raw['cleaned_context'] = df_raw['context'].apply(process_medical_context)

# Isolate only the columns we actually need, renaming for clarity
df_processed = df_raw[['pubid', 'question', 'cleaned_context', 'long_answer', 'final_decision']].copy()
df_processed.columns = ['pmid', 'question', 'context', 'ground_truth', 'label']

print("Cleaned DataFrame structure:")
df_processed.head(2)

Cleaned DataFrame structure:


,pmid,question,context,ground_truth,label
0,21645374,Do mitochondria play a role in remodelling lac...,Programmed cell death (PCD) is the regulated d...,Results depicted mitochondrial dynamics in viv...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,Assessment of visual acuity depends on the opt...,"Using the charts described, there was only a s...",no


In [5]:
# 1. Separate into Yes and No answers, discarding 'maybe'
df_yes = df_processed[df_processed['label'] == 'yes']
df_no = df_processed[df_processed['label'] == 'no']

# 2. Sample exactly 50 from each to build our 100-row baseline (random_state=42 ensures reproducibility)
df_yes_sample = df_yes.sample(n=50, random_state=42)
df_no_sample = df_no.sample(n=50, random_state=42)

# 3. Combine them back together and shuffle the deck
df_baseline = pd.concat([df_yes_sample, df_no_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final baseline shape: {df_baseline.shape}")
print("Label Distribution:")
print(df_baseline['label'].value_counts())

Final baseline shape: (100, 5)
Label Distribution:
label
no     50
yes    50
Name: count, dtype: int64


In [6]:
# Calculate basic text length statistics
df_baseline['question_word_count'] = df_baseline['question'].apply(lambda x: len(str(x).split()))
df_baseline['context_word_count'] = df_baseline['context'].apply(lambda x: len(str(x).split()))

print("--- Question Word Count Stats ---")
print(df_baseline['question_word_count'].describe().round(1))

print("\n--- Context Word Count Stats ---")
print(df_baseline['context_word_count'].describe().round(1))

--- Question Word Count Stats ---
count    100.0
mean      12.3
std        4.2
min        5.0
25%       10.0
50%       11.5
75%       15.0
max       29.0
Name: question_word_count, dtype: float64

--- Context Word Count Stats ---
count    100.0
mean     194.3
std       49.0
min       50.0
25%      163.0
50%      196.5
75%      228.0
max      350.0
Name: context_word_count, dtype: float64


In [7]:
# Save out to a local CSV in your Colab session
df_baseline.to_csv('raw_baseline.csv', index=False)
print("Saved raw_baseline.csv successfully!")

Saved raw_baseline.csv successfully!
